In [1]:
# Cell 1: Imports and helper functions

import os
import json
import pandas as pd
import numpy as np
from collections import Counter

# Percentile helper: ignores NaNs and returns None if series is empty
def pct(series, q):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return None
    return float(np.percentile(s, q, method="linear"))

# Error class extraction helper (gets the most common error_code)
def most_common_error(series):
    s = series.dropna()
    if s.empty:
        return None
    counts = Counter(s)
    return counts.most_common(1)[0][0] if counts else None

print("✓ Imports loaded successfully")

✓ Imports loaded successfully


In [2]:
# Cell 2: CONFIG — PASTE FULL PATHS ONLY

# ============================================================
# EDIT THESE THREE LINES - Paste complete paths
# ============================================================

# Database name (for display in output CSV)
db_name = "AtlasDB"

# Full path to the INPUT CSV file (writes.csv)
write_csv_path = r"C:\Users\avyaa\logs\writes.csv"

# Full path to the OUTPUT FOLDER (where you want write_metrics.csv saved)
# You must create this folder manually before running
write_out_dir = r"C:\Users\avyaa\Astra_results\Write\Data"

# ============================================================
# DO NOT EDIT BELOW
# ============================================================

# Output file paths
write_metrics_out = os.path.join(write_out_dir, "write_metrics.csv")
write_validation_out = os.path.join(write_out_dir, "write_validation.csv")
write_errors_out = os.path.join(write_out_dir, "write_errors_detail.csv")

# Write phase expectations
EXPECTED_RUNS_MIN = 100  # Approximately 100 runs per write type

# Verification
print("Database:", db_name)
print("Input CSV:", write_csv_path)
print("Output metrics:", write_metrics_out)
print("Output validation:", write_validation_out)
print("Output errors:", write_errors_out)

if not os.path.exists(write_csv_path):
    print("\n⚠️ INPUT FILE NOT FOUND! Check your path.")
else:
    print("\n✓ Input file found!")
    
if not os.path.exists(write_out_dir):
    print("⚠️ OUTPUT FOLDER NOT FOUND! Create it manually first.")
else:
    print("✓ Output folder exists!")

Database: AtlasDB
Input CSV: C:\Users\avyaa\logs\writes.csv
Output metrics: C:\Users\avyaa\Astra_results\Write\Data\write_metrics.csv
Output validation: C:\Users\avyaa\Astra_results\Write\Data\write_validation.csv
Output errors: C:\Users\avyaa\Astra_results\Write\Data\write_errors_detail.csv

✓ Input file found!
✓ Output folder exists!


In [3]:
# Cell 3: Load writes CSV and prep columns

# Required columns per logging strategy
required_cols = [
    "timestamp_utc","run_id","db_system","server_version","driver",
    "connection_params","db_name","phase","operation_type",
    "query_id","complexity","query_name","run_number",
    "execution_ms","rows_affected","returned_id",
    "error_code","error_message","params_json"
]

df = pd.read_csv(write_csv_path)

# Check for missing columns
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Keep only write phase operations
df = df[(df["phase"] == "writes") & (df["operation_type"] == "write")].copy()

# Normalize numerics
df["run_number"] = pd.to_numeric(df["run_number"], errors="coerce")
df["execution_ms"] = pd.to_numeric(df["execution_ms"], errors="coerce")
df["rows_affected"] = pd.to_numeric(df["rows_affected"], errors="coerce")

# Mark success/error rows
# Success = execution_ms is present and error_code is null/empty
df["success"] = df["execution_ms"].notna() & (
    df["error_code"].isna() | (df["error_code"].astype(str).str.strip() == "")
)
df["error"] = ~df["success"]

print("Write total rows:", len(df))
print("Success rows:", df["success"].sum())
print("Error rows:", df["error"].sum())

Write total rows: 1100
Success rows: 900
Error rows: 200


In [4]:
# Cell 4: Count validation (~100 executions per write type)

val_rows = []
for wid, g in df.groupby("query_id"):
    total_cnt = int(len(g))
    success_cnt = int(g["success"].sum())
    error_cnt = int(g["error"].sum())
    
    val_rows.append({
        "db_system": db_name,
        "query_id": wid,
        "total_runs": total_cnt,
        "success_count": success_cnt,
        "error_count": error_cnt,
        "expected_runs_min": EXPECTED_RUNS_MIN,
        "valid_counts": total_cnt >= EXPECTED_RUNS_MIN
    })

val_df = pd.DataFrame(val_rows).sort_values("query_id")
val_df.to_csv(write_validation_out, index=False)

print("✓ Validation saved to:", write_validation_out)
print("\nValidation summary:")
display(val_df)

✓ Validation saved to: C:\Users\avyaa\Astra_results\Write\Data\write_validation.csv

Validation summary:


,db_system,query_id,total_runs,success_count,error_count,expected_runs_min,valid_counts
0,AtlasDB,W1A,100,100,0,100,True
1,AtlasDB,W1B,100,100,0,100,True
2,AtlasDB,W1C,100,100,0,100,True
3,AtlasDB,W2,100,100,0,100,True
4,AtlasDB,W3A,100,0,100,100,True
5,AtlasDB,W3B,100,100,0,100,True
6,AtlasDB,W4,100,100,0,100,True
7,AtlasDB,W5,100,100,0,100,True
8,AtlasDB,W6,100,0,100,100,True
9,AtlasDB,W7A,100,100,0,100,True


In [5]:
# Cell 5: Percentiles per write type (p50, p95, p99, max) - success rows only

df_success = df[df["success"]].copy()

agg_write = []
for wid, g in df_success.groupby(["query_id","complexity","query_name"]):
    p50 = pct(g["execution_ms"], 50)
    p95 = pct(g["execution_ms"], 95)
    p99 = pct(g["execution_ms"], 99)
    mx  = float(np.max(g["execution_ms"])) if len(g) else None
    
    agg_write.append({
        "query_id": wid[0],
        "complexity": wid[1],
        "query_name": wid[2],
        "write_p50_ms": p50,
        "write_p95_ms": p95,
        "write_p99_ms": p99,
        "write_max_ms": mx,
        "success_count": int(len(g))
    })

write_df = pd.DataFrame(agg_write).sort_values("query_id")

print("✓ Write percentiles computed (success rows only)")
display(write_df.head(10))

✓ Write percentiles computed (success rows only)


,query_id,complexity,query_name,write_p50_ms,write_p95_ms,write_p99_ms,write_max_ms,success_count
0,W1A,Basic,Single-row insert (customer),190.6305,201.54010,220.14691,235.978,100
1,W1B,Basic,Single-row update (non-indexed),192.5345,225.34465,246.60422,260.981,100
2,W1C,Basic,Single-row delete,191.9090,203.08235,205.33726,230.707,100
3,W2,Moderate,Bulk insert (order_items),190.3685,201.10790,204.86540,206.093,100
4,W3B,Moderate,Conditional delete (cleanup),962.3790,1544.74715,1979.69913,2129.004,100
5,W4,Moderate,Upsert (LWT),195.9410,205.90275,212.31005,235.877,100
6,W5,Advanced,Transactional write mix (order),191.7755,203.23330,204.76393,218.122,100
7,W7A,Advanced,Update non-indexed (phone),197.6060,203.60890,230.01003,233.379,100
8,W7B,Advanced,Update indexed (email_lc),198.8405,202.71920,203.52094,203.713,100


In [6]:
# Cell 6: Error analysis per write type (handles zero errors)

df_errors = df[df["error"]].copy()

error_rows = []
for wid, g in df_errors.groupby(["query_id","complexity","query_name"]):
    error_cnt = int(len(g))
    
    # Most common error code
    common_code = most_common_error(g["error_code"])
    
    # Get one representative error message for that code
    rep_msg = None
    if common_code:
        msgs = g[g["error_code"] == common_code]["error_message"].dropna()
        rep_msg = msgs.iloc[0] if len(msgs) > 0 else None
    
    error_rows.append({
        "query_id": wid[0],
        "complexity": wid[1],
        "query_name": wid[2],
        "error_count": error_cnt,
        "most_common_error_code": common_code,
        "representative_error_message": rep_msg
    })

# Handle case where there are NO errors at all
if len(error_rows) > 0:
    error_df = pd.DataFrame(error_rows).sort_values("query_id")
else:
    # Create empty DataFrame with correct columns
    error_df = pd.DataFrame(columns=[
        "query_id","complexity","query_name","error_count",
        "most_common_error_code","representative_error_message"
    ])
    print("\n✓ No errors found in write phase!")

# Save detailed error info separately
error_df.to_csv(write_errors_out, index=False)

print("✓ Error details saved to:", write_errors_out)
if len(error_df) > 0:
    display(error_df)
else:
    print("  (No errors to display)")

✓ Error details saved to: C:\Users\avyaa\Astra_results\Write\Data\write_errors_detail.csv


,query_id,complexity,query_name,error_count,most_common_error_code,representative_error_message
0,W3A,Moderate,Conditional update (price range),100,InvalidRequest,Error from server: code=2200 [Invalid query] m...
1,W6,Advanced,Rollback (constraint violation),100,Exception,EmailAlreadyExists


In [7]:
# Cell 7: Merge percentiles, error counts, and validation (handles empty error_df)

# Start with percentiles
metrics = write_df.copy()

# Merge validation counts
metrics = pd.merge(
    metrics,
    val_df.loc[:, ["query_id","total_runs","error_count","valid_counts"]],
    on="query_id",
    how="left"
)

# Merge error details (only if error_df has data)
if len(error_df) > 0:
    metrics = pd.merge(
        metrics,
        error_df.loc[:, ["query_id","most_common_error_code"]],
        on="query_id",
        how="left"
    )
else:
    # No errors exist, add empty column
    metrics["most_common_error_code"] = None

# Calculate success rate
metrics["success_rate_pct"] = np.where(
    metrics["total_runs"] > 0,
    (metrics["success_count"] / metrics["total_runs"]) * 100.0,
    np.nan
)

# Attach db_system for clarity
metrics.insert(0, "db_system", db_name)

# Order columns
col_order = [
    "db_system","query_id","complexity","query_name",
    "total_runs","success_count","error_count","success_rate_pct","valid_counts",
    "write_p50_ms","write_p95_ms","write_p99_ms","write_max_ms",
    "most_common_error_code"
]
metrics = metrics.loc[:, col_order]

# Save
metrics.to_csv(write_metrics_out, index=False)

print("✓ Metrics saved to:", write_metrics_out)
print("\nFinal Write metrics:")
display(metrics)

✓ Metrics saved to: C:\Users\avyaa\Astra_results\Write\Data\write_metrics.csv

Final Write metrics:


,db_system,query_id,complexity,query_name,total_runs,success_count,error_count,success_rate_pct,valid_counts,write_p50_ms,write_p95_ms,write_p99_ms,write_max_ms,most_common_error_code
0,AtlasDB,W1A,Basic,Single-row insert (customer),100,100,0,100.0,True,190.6305,201.54010,220.14691,235.978,NaN
1,AtlasDB,W1B,Basic,Single-row update (non-indexed),100,100,0,100.0,True,192.5345,225.34465,246.60422,260.981,NaN
2,AtlasDB,W1C,Basic,Single-row delete,100,100,0,100.0,True,191.9090,203.08235,205.33726,230.707,NaN
3,AtlasDB,W2,Moderate,Bulk insert (order_items),100,100,0,100.0,True,190.3685,201.10790,204.86540,206.093,NaN
4,AtlasDB,W3B,Moderate,Conditional delete (cleanup),100,100,0,100.0,True,962.3790,1544.74715,1979.69913,2129.004,NaN
5,AtlasDB,W4,Moderate,Upsert (LWT),100,100,0,100.0,True,195.9410,205.90275,212.31005,235.877,NaN
6,AtlasDB,W5,Advanced,Transactional write mix (order),100,100,0,100.0,True,191.7755,203.23330,204.76393,218.122,NaN
7,AtlasDB,W7A,Advanced,Update non-indexed (phone),100,100,0,100.0,True,197.6060,203.60890,230.01003,233.379,NaN
8,AtlasDB,W7B,Advanced,Update indexed (email_lc),100,100,0,100.0,True,198.8405,202.71920,203.52094,203.713,NaN


In [8]:
# Cell 8: Sanity checks and summaries

print("=" * 60)
print("WRITE METRICS COMPUTATION COMPLETE")
print("=" * 60)

print("\n📁 Files saved:")
print("  -", write_metrics_out)
print("  -", write_validation_out)
print("  -", write_errors_out)

print("\n⚠️ Write types missing expected counts:")
invalid = val_df[~val_df["valid_counts"]]
if len(invalid) > 0:
    display(invalid)
else:
    print("  ✓ All write types have valid counts!")

print("\n⚠️ Write types with errors:")
with_errors = metrics[metrics["error_count"] > 0].sort_values("error_count", ascending=False)
if len(with_errors) > 0:
    display(with_errors)
else:
    print("  ✓ No errors recorded!")

print("\n⚡ Slowest write operations (top 5 by p99):")
display(metrics.sort_values("write_p99_ms", ascending=False).head(5))

print("\n📊 Success rate summary:")
print(f"  Average success rate: {metrics['success_rate_pct'].mean():.2f}%")
print(f"  Min success rate: {metrics['success_rate_pct'].min():.2f}%")
print(f"  Max success rate: {metrics['success_rate_pct'].max():.2f}%")

WRITE METRICS COMPUTATION COMPLETE

📁 Files saved:
  - C:\Users\avyaa\Astra_results\Write\Data\write_metrics.csv
  - C:\Users\avyaa\Astra_results\Write\Data\write_validation.csv
  - C:\Users\avyaa\Astra_results\Write\Data\write_errors_detail.csv

⚠️ Write types missing expected counts:
  ✓ All write types have valid counts!

⚠️ Write types with errors:
  ✓ No errors recorded!

⚡ Slowest write operations (top 5 by p99):


,db_system,query_id,complexity,query_name,total_runs,success_count,error_count,success_rate_pct,valid_counts,write_p50_ms,write_p95_ms,write_p99_ms,write_max_ms,most_common_error_code
4,AtlasDB,W3B,Moderate,Conditional delete (cleanup),100,100,0,100.0,True,962.3790,1544.74715,1979.69913,2129.004,NaN
1,AtlasDB,W1B,Basic,Single-row update (non-indexed),100,100,0,100.0,True,192.5345,225.34465,246.60422,260.981,NaN
7,AtlasDB,W7A,Advanced,Update non-indexed (phone),100,100,0,100.0,True,197.6060,203.60890,230.01003,233.379,NaN
0,AtlasDB,W1A,Basic,Single-row insert (customer),100,100,0,100.0,True,190.6305,201.54010,220.14691,235.978,NaN
5,AtlasDB,W4,Moderate,Upsert (LWT),100,100,0,100.0,True,195.9410,205.90275,212.31005,235.877,NaN



📊 Success rate summary:
  Average success rate: 100.00%
  Min success rate: 100.00%
  Max success rate: 100.00%
